# CLIP for Medical Imaging

1. CLIP is trained on natural images + text captions (web-scale data)
2. CLIP's knowledge of medical images (X-rays, MRIs, CT scans) is very limited.
3. CLIP produces embeddings; it's doesn't classify unless you create a prompt-based zero-shot setup.


We can use CLIP for semantic search, similarity retrieval, or zero-shot classification, but accuracy for medical diagnosis may be unreliable unless fine-tuned.

In [ ]:
# install the libraries

!pip install torch torchvision
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-l4cisu60
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-l4cisu60
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.0 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=5298c11e417d49dde0c21b3f088102e9efaf74ff94a1fa0aa9ea4ba037a79502
  Stored in directory: /tmp/pip-ephem-wheel-cache-3aedns88/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [ ]:
import torch, clip
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

100%|███████████████████████████████████████| 338M/338M [00:05<00:00, 62.3MiB/s]


In [ ]:
model.eval()

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): Sequential(
        (0): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): QuickGELU()
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
        (1): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          

In [ ]:
# load the medical image

# img = preprocess(Image.open("xray_image.jpeg")).unsqueeze(0).to(device)
img = preprocess(Image.open("pneumonia.jpeg")).unsqueeze(0).to(device)

In [ ]:
# define labels and prompts

labels = ["normal chest X-ray", "pneumonia", "lung cancer"]
texts = [f"a {label}" for label in labels]
text_tokens = clip.tokenize(texts).to(device)

In [ ]:
# encode image and text

with torch.no_grad():
    img_feat = model.encode_image(img)
    txt_feat = model.encode_text(text_tokens)

# Normalize
img_feat /= img_feat.norm(dim=-1, keepdim=True)
txt_feat /= txt_feat.norm(dim=-1, keepdim=True)

In [ ]:
# compute similarity or clasification

similarity = (img_feat @ txt_feat.T).softmax(dim=-1)
for label, score in zip(labels, similarity[0]):
    print(label, float(score))

normal chest X-ray 0.3326646089553833
pneumonia 0.3306644558906555
lung cancer 0.3366709053516388
